In [1]:
import event_helpers
import gcsfs
import geopandas as gpd
import google.auth
import pandas as pd
import poi_geography_utils as poi_geog_utils
import world_cup_vars as wc_vars
from gtfs_curator_utils import utils

GCS_FILE_PATH = wc_vars.GCS_FILE_PATH
credentials, _ = google.auth.default()

In [2]:
STADIUM_NAME = "sofi"
stop_gdf = gpd.read_parquet(
    f"{GCS_FILE_PATH}stop_summary_{STADIUM_NAME}.parquet",
    storage_options = {"token": credentials.token}
)

In [3]:
stop_gdf.dtypes

schedule_name                         object
stop_id                               object
stop_name                             object
route_id_array                        object
route_type_array                      object
geometry                            geometry
daily_arrivals_weekday_non_event     float64
daily_arrivals_weekend_non_event     float64
daily_arrivals_weekday_event         float64
daily_arrivals_weekend_event         float64
change_daily_arrivals_weekday        float64
change_daily_arrivals_weekend        float64
combined_change_daily_arrivals       float64
near_route_ids                        object
special_route_ids                     object
is_route_near                           bool
is_special_route                        bool
is_stop_near                            bool
point_of_interest                     object
dtype: object

In [4]:
import create_route_and_stop_event_summary as A1

routes_df = A1.import_routes_near_poi(wc_vars.event_name, "sofi")

stop_gdf = A1.prep_fct_daily_scheduled_stops(wc_vars.event_name, wc_vars.sofi_match_times).merge(
    routes_df[["feed_key", "schedule_name"]].drop_duplicates(),
    how="inner",
)


In [ ]:
stop_gdf.dtypes

In [5]:
time_of_day = "midday"
day_type = "weekday"

filtered_sofi_dict = event_helpers.grab_matches_by_day_type(
    wc_vars.sofi_match_times, time_of_day, day_type)
filtered_sofi_dict

{'2026-06-18': 'midday', '2026-07-02': 'midday', '2026-07-10': 'midday'}

In [6]:
stop_gdf.dtypes

service_date                  datetime64[ns]
feed_key                              object
stop_id                               object
stop_name                             object
daily_arrivals                         Int64
geometry                            geometry
arrivals_per_hour_owl                float64
arrivals_per_hour_early_am           float64
arrivals_per_hour_am_peak            float64
arrivals_per_hour_midday             float64
arrivals_per_hour_pm_peak            float64
arrivals_per_hour_evening            float64
arrivals_owl                           Int64
arrivals_early_am                      Int64
arrivals_am_peak                       Int64
arrivals_midday                        Int64
arrivals_pm_peak                       Int64
arrivals_evening                       Int64
route_id_array                        object
route_type_array                      object
event_day                               bool
day_type                              object
event_time

In [7]:
# get non-event 
nonevent_df = stop_gdf[~stop_gdf.service_date.isin(wc_vars.sofi_dates)]

# for event, get individual dates
event_df = stop_gdf[stop_gdf.service_date.isin(wc_vars.sofi_dates)]

/tmp/ipykernel_2897/304258115.py:2: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  nonevent_df = stop_gdf[~stop_gdf.service_date.isin(wc_vars.sofi_dates)]
/tmp/ipykernel_2897/304258115.py:5: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  event_df = stop_gdf[stop_gdf.service_date.isin(wc_vars.sofi_dates)]


In [10]:
# how should comparison be done, grab those dates out and grab all the non-event dates? put those into aggregation?
arrivals_cols = [c for c in stop_gdf.columns if "arrivals_per_hour" in c]

arrivals_by_event_df = (
    event_helpers.aggregate_by_event_type(
        nonevent_df,
        group_cols=[
            "schedule_name",
            "stop_id",
            "stop_name",
            "event_day",
            "day_type",
        ],
        mean_cols=arrivals_cols
    ).pipe(event_helpers.merge_in_stop_geom, stop_gdf)
)

In [14]:
# weekday, midday matches
filtered_sofi_dict

{'2026-06-18': 'midday', '2026-07-02': 'midday', '2026-07-10': 'midday'}

In [29]:
def make_long(df, id_vars: list):
    df_long = pd.melt(df,
        id_vars = id_vars,
        value_vars = [f"arrivals_per_hour_{t}" for t in ["early_am", "am_peak", "midday", "pm_peak", "evening", "owl"]],
        var_name = "time_of_day",
        value_name = "arrivals_per_hour"
    )

    df_long = df_long.assign(
        time_of_day = df_long.time_of_day.str.replace('arrivals_per_hour_', '')
    )

    return df_long
    
event_df_long = make_long(
    event_df[event_df.service_date.isin(filtered_sofi_dict.keys())], 
    id_vars = ["service_date", "schedule_name", "stop_id", "stop_name"]
)

/tmp/ipykernel_2897/859050242.py:16: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  event_df[event_df.service_date.isin(filtered_sofi_dict.keys())],


In [31]:
nonevent_long = make_long(
    arrivals_by_event_df[arrivals_by_event_df.day_type == "weekday"],
    id_vars = ["schedule_name", "stop_id", "stop_name"]
)

In [39]:
nonevent_long2 = nonevent_long.groupby(["schedule_name", "time_of_day"]).agg(
    {
        "arrivals_per_hour": "mean",
        "stop_id": "count"
    }).reset_index()

In [50]:
import altair as alt

alt.data_transformers.enable("vegafusion")

non_event_chart = alt.Chart(nonevent_long2).mark_bar().encode(
    x="arrivals_per_hour",
    y=alt.Y("time_of_day", 
            sort=["early_am", "am_peak", "midday", "pm_peak", "evening", "owl"]
           ),
)

In [56]:
event_df_long2 = event_df_long.groupby(["service_date","schedule_name", "time_of_day"]).agg(
    {
        "arrivals_per_hour": "mean",
        "stop_id": "count"
    }).reset_index()


event_jul10_chart = alt.Chart(event_df_long2[event_df_long2.service_date=="2026-07-10"]).mark_bar().encode(
    x="arrivals_per_hour",
    y=alt.Y("time_of_day", 
            sort=["early_am", "am_peak", "midday", "pm_peak", "evening", "owl"]
           ),
    color=alt.value("purple")
)


event_jul2_chart = alt.Chart(event_df_long2[event_df_long2.service_date=="2026-07-02"]).mark_bar().encode(
    x="arrivals_per_hour",
    y=alt.Y("time_of_day", 
            sort=["early_am", "am_peak", "midday", "pm_peak", "evening", "owl"]
           ),
    color=alt.value("orange")
)

event_jun18_chart = alt.Chart(event_df_long2[event_df_long2.service_date=="2026-06-18"]).mark_bar().encode(
    x="arrivals_per_hour",
    y=alt.Y("time_of_day", 
            sort=["early_am", "am_peak", "midday", "pm_peak", "evening", "owl"]
           ),
    color=alt.value("green")
)

In [59]:
alt.vconcat(
    non_event_chart, event_jun18_chart, 
    event_jul2_chart, event_jul10_chart
).resolve_scale(x="shared")

alt.VConcatChart(...)